# Break Through Tech AI: Nestlé 1A Group
## Stage 1: Building the DataFrame

As of 09/06/2025, the Nestlé 1A group has decided to use a subset of the [Amazon Reviews](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023) dataset collected in 2023 by McAuley Lab. The entire dataset consists of 571.54 M examples. We are using the raw data from the "Grocery and Gourmet Food" category, which consists of over 14 M examples, to build an initial dataframe to then preprocess and develop machine learning models from.

Note: Outputs have been cleared due to rendering issues. Please run on your personal machine.

#### Step 0. Update, Install, and Import Python Libraries

In [53]:
# !pip install --upgrade pip
# !pip install -q datasets huggingface_hub pyarrow pandas
# !pip install matplotlib
# !pip install seaborn
# !pip install scikit-learn
# !pip install seaborn

In [57]:
from huggingface_hub import hf_hub_download
from datasets import load_dataset
from tqdm import tqdm
import gc
import json
import pandas as pd
import pyarrow as pa
import pickle as pkl

#### Step 1. Upload User Reviews file from HuggingFace

In [3]:
REV_PATH = "raw/review_categories/Grocery_and_Gourmet_Food.jsonl"

In [4]:
rev_file = hf_hub_download(repo_id="McAuley-Lab/Amazon-Reviews-2023", filename=REV_PATH, repo_type="dataset",)

In [5]:
ds_rev = load_dataset("json", data_files=rev_file, split="train")

In [6]:
# Note: This cell may take a while to run.
df_rev = ds_rev.to_pandas()

##### Data Fields for User Reviews

| Field            | Type   | Explanation |
| :--------------- | :----- | :---------- |
| rating           | float  | Rating of the product (from 1.0 to 5.0). |
| text             | str    | Text body of the user review. |
| images           | list   | Images that users post after they have received the product.<br><br>Note: Each image has different sizes (small, medium, large), represented by the `small_image_url`, `medium_image_url`, and `large_image_url` respectively. |
| asin             | str    | ID of the product. |
| parent_asin      | str    | Parent ID of the product.<br><br>Note: Products with different colors, styles, sizes usually belong to the same parent ID. The “asin” in previous Amazon datasets is actually parent ID. Please use parent ID to find product meta. |
| user_id          | str    | ID of the reviewer. |
| timestamp        | int    | Time of the review (unix time). |
| verified_purchase| bool   | User purchase verification. |
| helpful_vote     | int    | Helpful votes of the review. |

#### Step 2. Upload Item Metadata file from HuggingFace

In [7]:
META_PATH = "raw/meta_categories/meta_Grocery_and_Gourmet_Food.jsonl"

In [8]:
# Filters down to only products with reviews
needed_parent_asin = set(df_rev["parent_asin"].unique())

In [9]:
# Loading metadata
meta_file = hf_hub_download(
    repo_id="McAuley-Lab/Amazon-Reviews-2023",
    filename=META_PATH,
    repo_type="dataset",
)


In [ ]:
# Selected few columns from metadata
meta_rows = []
with open(meta_file, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Parsing meta file"):
        obj = json.loads(line)
        row = {
            "parent_asin": obj.get("parent_asin"),
            "title_meta": obj.get("title"),
            "main_category": obj.get("main_category"),
            "average_rating": obj.get("average_rating"),
            "rating_number": obj.get("rating_number"),
            "price": obj.get("price"),
            "details": obj.get("details")  # keep raw JSON dict for now
        }
        # Filters down to only products with reviews
        if row["parent_asin"] not in needed_parent_asin:
            continue
        meta_rows.append(row)

In [11]:
df_meta = pd.DataFrame(meta_rows)

##### Data Fields for Item Metadata

| Field           | Type  | Explanation |
| :--------------- | :---- | :---------- |
| main_category    | str   | Main category (i.e., domain) of the product. |
| title            | str   | Name of the product. |
| average_rating   | float | Rating of the product shown on the product page. |
| rating_number    | int   | Number of ratings in the product. |
| features         | list  | Bullet-point format features of the product. |
| description      | list  | Description of the product. |
| price            | float | Price in US dollars (at time of crawling). |
| images           | list  | Images of the product. Each image has different sizes (thumb, large, hi_res). The “variant” field shows the position of the image. |
| videos           | list  | Videos of the product, including title and URL. |
| store            | str   | Store name of the product. |
| categories       | list  | Hierarchical categories of the product. |
| details          | dict  | Product details, including materials, brand, sizes, etc. |
| parent_asin      | str   | Parent ID of the product. |
| bought_together  | list  | Recommended bundles from the website. |


#### Step 3. Data Cleaning: Remove Duplicates, Handle Missing Values, Only Include Verified Purchases


##### Step 3a. User Reviews DataFrame

In [ ]:
print("Shape before dropping duplicates:", df_rev.shape)

In [ ]:
# Check for duplicate reviews based on user_id, asin, and text
dup_count = df_rev.duplicated(subset=["user_id", "asin", "text"]).sum()
print("Total duplicate reviews:", dup_count)

In [ ]:
# Drop duplicate reviews
df_rev = df_rev.drop_duplicates(subset=["user_id", "asin", "text"])
print("Shape after dropping duplicates:", df_rev.shape)

In [ ]:
# Count missing ratings
print("Missing ratings:", df_rev['rating'].isna().sum())

In [16]:
# Remove missing or empty review text
df_rev = df_rev.dropna(subset=['text'])
df_rev = df_rev[df_rev['text'].str.strip() != '']

In [ ]:
# Count missing helpful_vote
print("Missing helpful_vote:", df_rev['helpful_vote'].isna().sum())

In [ ]:
# Count missing verified_purchase
print("Missing verified_purchase:", df_rev['verified_purchase'].isna().sum())

In [ ]:
df_rev['verified_purchase'].value_counts()

In [21]:
df_rev = df_rev[df_rev['verified_purchase'] == True]

In [ ]:
print("Final df_rev shape:", df_rev.shape)

##### Step 3b. Item Metadata DataFrame

In [ ]:
print("Initial df_meta shape:", df_meta.shape)

In [ ]:
print("Missing ratings:", df_meta['average_rating'].isna().sum())

#### Step 4. Merge the User Reviews and Item Metadata DataFrames into one

In [30]:
df = df_rev.merge(
    df_meta,
    on="parent_asin", how="left"
)

#### Step 5. Convert the file for Data Preprocessing

In [ ]:
# Export merged dataframe as parquet chunks

# chunk_size = 1000000  # 1M records per chunk
# total_chunks = len(df) // chunk_size + 1

# for i in tqdm(range(0, len(df), chunk_size)):
#     chunk = df.iloc[i:i+chunk_size]
#     chunk['price'] = pd.to_numeric(df['price'].replace('—', np.nan), errors='coerce')
#     chunk.to_parquet(f'df_chunk_{i//chunk_size:03d}.parquet', index=False)
#     del chunk
#     gc.collect()
    
# print(f"Exported {total_chunks} parquet chunks")

#### Step 6. Flavor Analysis

In [ ]:
# View flavors from 'details' column and their mention counts 
df["details"].str.get("Flavor").value_counts()

In [ ]:
df['timestamp'].head(10)

In [34]:
# Convert timestamps to actual date&time format
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')


In [ ]:
df['timestamp'].head()

In [ ]:
# Create year_quarter column in df
df['year_quarter'] = df['timestamp'].dt.to_period('Q')
df[['timestamp', 'year_quarter']].head()


In [ ]:
# View first row of 'details' column
df['details'].iloc[0]

In [ ]:
# Define a list of generic flavor terms to exclude
generic_flavors = ['Original', 'Assorted', 'Variety', 'Mixed', 'Classic', 'Regular', 'Plain']

df['Flavor'] = df['details'].apply(lambda x: x.get('Flavor') if isinstance(x, dict) and 'Flavor' in x else None)

# Drop rows with generic or missing flavors
df = df[~df['Flavor'].isin(generic_flavors)]
df = df.dropna(subset=['Flavor'])

# View flavor and its number of mentions
df['Flavor'].value_counts().head(10)

In [ ]:
df['Flavor'].head(20)

In [ ]:
df['Flavor'].isna().sum() 

In [ ]:
total_rows = len(df)
flavor_rows = df['Flavor'].notna().sum()
print(f"Keeping {flavor_rows/total_rows:.2%} of rows with valid flavor info")

In [ ]:
df['Flavor'].notna().sum()

In [43]:
# find the top mentioned flavors within our dataframe
top_flavors = (df['Flavor'].value_counts().head(100).index)
# Create a dataframe for top flavors
df_top = df[df['Flavor'].isin(top_flavors)].copy()

In [44]:
# Convert timestamp to the correct date&time format
df_top['timestamp'] = pd.to_datetime(df_top['timestamp'], unit='ms', errors='coerce')
# Create quarter column based on timestamps
df_top['year_quarter'] = df_top['timestamp'].dt.to_period('Q')

In [ ]:
df_top[['timestamp', 'year_quarter']].head()

In [46]:
# Group data by flavors and quarter
trend_df = (df_top.groupby(['year_quarter', 'Flavor']).size().reset_index(name='mention_count'))


In [47]:
# Sort flavor quarters in chronological order
trend_df = trend_df.sort_values(by=['Flavor', 'year_quarter'])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

top25 = df_top['Flavor'].value_counts().head(25).index

for flavor in top25:
    subset = trend_df[trend_df['Flavor'] == flavor].sort_values('year_quarter')
    plt.plot(subset['year_quarter'].dt.to_timestamp(),   # ensures correct order
             subset['mention_count'],
             label=flavor)

plt.title("Top 25 Flavors Mentioned Quarterly", fontsize=14)
plt.xlabel("Quarter", fontsize=12)
plt.ylabel("Mentions", fontsize=12)

# Rotate and shrink x-axis labels
plt.xticks(rotation=45, ha='right', fontsize=8)

# Move legend outside the plot
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', ncol=2, fontsize=9, frameon=False)

# Use tight layout to fit everything neatly 
plt.tight_layout()
plt.show()


In [49]:
# Compute growth rate quarter over quarter
trend_df['growth_rate'] = trend_df.groupby('Flavor')['mention_count'].pct_change()

In [ ]:
trend_df.head(10)

In [51]:
# Find most consistently growing flavors
top_growing = trend_df.groupby('Flavor')['growth_rate'].mean().sort_values(ascending=False)

In [ ]:
# Calculate top 25 flavors with highest growth rate
top_growing = trend_df.groupby('Flavor')['growth_rate'].mean().sort_values(ascending=False).head(25)
top_growing.head(25)

In [ ]:
df.shape

In [ ]:
df_top.shape

In [ ]:
df_top.to_pickle('df_top.pkl')